In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import anndata as ad
import sys
import pandas as pd
from statannotations.Annotator import Annotator
from matplotlib.colors import LinearSegmentedColormap
import scipy
import seaborn as sns
from scipy import spatial
from sklearn.neighbors import NearestNeighbors

In [ ]:
from typing import List, Optional

In [ ]:
%matplotlib inline
# create the plots suitable for modification in adobe Illustrator
plt.rcParams['pdf.fonttype'] = 42 

In [ ]:
from matplotlib import font_manager
print(sorted([f.name for f in font_manager.fontManager.ttflist]))

# Define global functions and variables

In [ ]:
figs = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/figures"
obj = "/public/home/liwang/project/lung_cancer_ST/scRNA_data/young_lung_cancer_collection/analysis_health_tumor_v6/objects"

In [ ]:
def filter_spots(adata, celltype, fraction_cutoff=None, quantile_cutoff=None):
    """
    filter the spots based on the deconvolution results
    
    Args:
        fraction_cutoff (doubel): only the spots with the deconvolution fraction above this fraction cutoff is considered as celltype spots. Value is [0-1]

        quantile_cutoff (doubel): only the spots with the deconvolution fraction above this quantile cutoff is considered as celltype spots. Value is [0-1]
        
        celltype (string): the column represting decomvolution results in adata.obs
        
    Returns:
        A adata objects with filtered celltype spots
    """
    if fraction_cutoff is not None and quantile_cutoff is not None:
        raise ValueError("fraction_cutoff and quantile_cutoff are mutually exclusive. Please set only one of them.")

    
    if fraction_cutoff is not None:
        cutoff = fraction_cutoff
    elif quantile_cutoff is not None:
        cutoff = np.quantile(a=adata.obs[celltype], q=quantile_cutoff)
    else:
        raise ValueError("At least one of fraction_cutoff or quantile_cutoff must be set.")

    adata_subset = adata[adata.obs[celltype] > cutoff].copy()
    
    return adata_subset

In [ ]:
def distance_measurement (adata: ad,
                          slices_column: str,
                          slices: str, 
                          celltype1: str,
                          celltype2: str,
                          type1_spots_cutoff: str, 
                          type2_spots_cutoff: str):
    '''
    get the min distance between celltype1 spots and celltype2 spot. the distances are celltype1-based
    
    Args:
        slices_column: the name of column contains the slices info
        
        slices: the chosed slice
        
        celltype1: celltype1
        
        celltype2: celltype2
        
        type1_spots_cutoff: only the spots with the deconvolution fraction above this cutoff is considered as celltype spots. Value is [0-1]
        
        type2_spots_cutoff: only the spots with the deconvolution fraction above this cutoff is considered as celltype spots. Value is [0-1]

    Ruturns:
        A numpy.ndarray with the shape (#celltype1 spots, )
    '''
    
    slice_h5ad = adata[adata.obs[slices_column] == slices]
    
    type1_spots = filter_spots(adata=slice_h5ad, celltype=celltype1, fraction_cutoff=type1_spots_cutoff)
    type2_spots = filter_spots(adata=slice_h5ad, celltype=celltype2, fraction_cutoff=type2_spots_cutoff)
    
    type1_spots_coord = type1_spots.obsm['spatial']
    type2_spots_coord = type2_spots.obsm['spatial']
    
    spots_distance = spatial.distance.cdist(XA=type1_spots_coord, XB=type2_spots_coord, metric='euclidean')
    
    spots_distance_min = np.min(a=spots_distance, axis=1)
    
    return spots_distance_min

In [ ]:
def neighbor_measurement (adata: ad,
                          slices_column: str,
                          slices: str, 
                          base_cell_type: str,
                          query_cell_type: list[str],
                          base_cell_type_spots_fraction_cutoff: float = None,
                          base_cell_type_spots_quantile_cutoff: float = None):
    '''
    get the query_cell_type deconvolution fraction in nearby spots of base_cell_type
    
    Args:
        slices_column: the name of column contains the slices info
        
        slices: the chosed slice
        
        base_cell_type: base_cell_type
        
        query_cell_type: query_cell_type
        
        base_cell_type_spots_fraction_cutoff:
        only the spots with the deconvolution fraction above this fraction cutoff is considered as celltype spots. Value is [0-1]

        base_cell_type_spots_quantile_cutoff:
        only the spots with the deconvolution fraction above this qunatile cutoff is considered as celltype spots. Value is [0-1]


    Ruturns:
        A pd.DataFrame with query_cell_type deconvolution fraction and base_cell_type
    '''
    
    slice_h5ad = adata[adata.obs[slices_column] == slices]
    
    base_cell_type_spots = filter_spots(adata=slice_h5ad, celltype=base_cell_type,
                                        fraction_cutoff=base_cell_type_spots_fraction_cutoff,
                                        quantile_cutoff=base_cell_type_spots_quantile_cutoff)
    
    spots_distance = spatial.distance.cdist(XA=base_cell_type_spots.obsm['spatial'], XB=slice_h5ad.obsm['spatial'], metric='euclidean')
    
    nearby_spots = {}
    for i in range(spots_distance.shape[0]):
        condition = spots_distance[i] < 210 #unit distance ~200
        indices = np.where(condition)[0]
        nearby_spots[i] = indices.tolist()
        
    query_cell_type_frac = {i:pd.DataFrame.apply(slice_h5ad.obs.iloc[nearby_spots[i], :][query_cell_type],axis=0, func=np.mean) for i in range(len(nearby_spots))}
    query_cell_type_frac = pd.DataFrame(query_cell_type_frac.values())
    query_cell_type_frac['Slice'] = slices
    query_cell_type_frac['Base_cell_type'] = base_cell_type
    
    return query_cell_type_frac

In [ ]:
#spointed_samples_concated_maincluster = sc.read_h5ad(f"{obj}/ST_deconvolution_obj/spointed_samples_concated_maincluster.h5ad")
#spointed_samples_concated_subcluster = sc.read_h5ad(f"{obj}/ST_deconvolution_obj/spointed_samples_concated_subcluster.h5ad")

In [ ]:
#spointed_samples_concated_subcluster

In [ ]:
# Load data
all_8samples_st_SCT_ssGSEA = sc.read_h5ad(f"{obj}/all_8samples_st_ssGSEA_SCT_data.h5ad")
all_8samples_st_SpatialCounts_ssGSEA = sc.read_h5ad(f"{obj}/all_8samples_st_ssGSEA_Spatial_counts.h5ad")

In [ ]:
# merge the SCT normalized counts and SpatialCounts
all_8samples_st_ssGSEA = all_8samples_st_SCT_ssGSEA.copy()
all_8samples_st_ssGSEA.raw = all_8samples_st_SpatialCounts_ssGSEA.copy()

#set the obsm.['spatial'] coord for plotting
all_8samples_st_ssGSEA.obsm['spatial'] = np.array(all_8samples_st_ssGSEA.obs[['imagecol', 'imagerow']])

# All_8samples_st_analysis

## TLS Region analysis

### TLS Visualization

In [ ]:
all_8samples_st_ssGSEA.obs['TLS_Region_ssGSEA']

In [ ]:
sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == 'LUAD_A'],
                        color=['TLS_signature_from_Dieu_Trends_Immunol_2014'],
                        basis = 'spatial',
                        size=40, vmin=0,vmax=0.5,
                        add_outline=True, frameon = True, show=False, title='TLS Score');


In [ ]:
sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == 'TD8'],
                        color=['TLS_Region_ssGSEA'],
                        basis = 'spatial',
                        size=40,
                        add_outline=True, frameon = True, show=False, title='TLS Region');

In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(2,4, figsize=(18,8), dpi=300)
patients = [['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3'], ['TD1', 'TD5', 'TD6', 'TD8']]
#celltype_vmax ={'Naive_B':0.2, 'Memory_B':0.4, 'CXCL13_CD4_Tex':0.1, 'CXCL13_CD8_Tex':0.1}

for i in range(2):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patients[i][j]],
                        color=['TLS_signature_from_Dieu_Trends_Immunol_2014'],
                        basis = 'spatial',
                        title=f'{patients[i][j]}; TLS Score',
                        size=40, vmin=0, vmax=0.5,
                        add_outline=True, frameon = True,
                        show=False, ax=axs[i][j]);
        
        axs[i][j].invert_yaxis();
        axs[i][j].set_facecolor('none');
        
Fig.patch.set_facecolor('none')
Fig.savefig(f'{figs}/Figures_raw/ST_TLS_Score_Spatial_ScatterPlot.pdf', bbox_inches='tight', transparent = True)


In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(2,4, figsize=(18,8), dpi=300)
patients = [['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3'], ['TD1', 'TD5', 'TD6', 'TD8']]
#celltype_vmax ={'Naive_B':0.2, 'Memory_B':0.4, 'CXCL13_CD4_Tex':0.1, 'CXCL13_CD8_Tex':0.1}

for i in range(2):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patients[i][j]].copy(),
                        color=['TLS_Region_ssGSEA'],
                        basis = 'spatial',
                        title=f'{patients[i][j]}; TLS Region',
                        size=40, 
                        add_outline=True, frameon = True,
                        show=False, ax=axs[i][j]);
        
        axs[i][j].invert_yaxis();
        axs[i][j].set_facecolor('none');
        
Fig.patch.set_facecolor('none')
Fig.savefig(f'{figs}/Figures_raw/ST_TLS_Region_Spatial_ScatterPlot.pdf', bbox_inches='tight', transparent = True)

## CXCL13_T_B analysis

### Spatial visualization

In [ ]:
# Load the RCTD resultes (cluster)
RCTD_maincluster_weights_df = pd.read_csv(f"{obj}/RCTD_FullMode_diseased_maincluster_weights_df.csv")
RCTD_maincluster_weights_df.set_index(keys = 'barcode', inplace = True)

all_8samples_st_ssGSEA.obs = all_8samples_st_ssGSEA.obs.join(other=RCTD_maincluster_weights_df)

In [ ]:
all_8samples_st_ssGSEA

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(14, 8), dpi=300)

samples = ['LUAD_A']
celltype = ['Tumor', 'T_Cell', 'B_Cell',
            'Macro', 'Endo', 'Fibro']
celltype_vmax = {'Tumor': 0.4, 'T_Cell': 0.4, 'B_Cell': 0.4,
                 'Macro': 0.5, 'Endo': 0.5, 'Fibro': 0.5}

for i in range(2):
    for j in range(3):
        ct = celltype[i * 3 + j]                      # 关键：把二维下标映射到一维列表
        adata_smp = all_8samples_st_ssGSEA[
            all_8samples_st_ssGSEA.obs['orig.ident'] == samples[0]
        ].copy()

        sc.pl.embedding(
            adata_smp,
            color=ct,
            basis='spatial',
            size=40,
            vmax=celltype_vmax[ct],
            add_outline=True,
            frameon=True,
            show=False,
            ax=axs[i][j],
        )

        ax = axs[i][j]
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title(ct, fontsize=12)   # 建议保留细胞类型名，否则 6 张图分不清
        ax.invert_yaxis()

plt.tight_layout()
plt.show()

fig.savefig(f'{figs}/Figures_raw/Maincluster_deconv_Tumor_T_B_Macro_Endo_Fibro_ScatterPlot.pdf', bbox_inches='tight')


In [ ]:
96.181/351.621

In [ ]:
# Load the RCTD resultes (subcluster)
RCTD_subcluster_weights_df = pd.read_csv(f"{obj}/RCTD_FullMode_diseased_subcluster_weights_df.csv")
RCTD_subcluster_weights_df.set_index(keys = 'barcode', inplace = True)

all_8samples_st_ssGSEA.obs = all_8samples_st_ssGSEA.obs.join(other=RCTD_subcluster_weights_df, rsuffix='_SubClus')

In [ ]:
all_8samples_st_ssGSEA

In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(6,4, figsize=(18,27), dpi=300)
samples = ['LUAD_B', 'LUAD_C', 'TD3', 'TD1', 'TD6', 'TD8']
celltype = ['CXCL13_CD4_Tex', 'CXCL13_CD8_Tex', 'Naive_B', 'Memory_B']
celltype_vmax ={'Naive_B':0.15, 'Memory_B':0.15, 'CXCL13_CD4_Tex':0.05, 'CXCL13_CD8_Tex':0.05}

for i in range(6):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == samples[i]].copy(),
                        color=celltype[j],
                        basis = 'spatial',
#                        color_map='inferno',
                        size=40, vmax=celltype_vmax[celltype[j]],
                        add_outline=True, frameon = True, show=False, ax=axs[i][j]);

        
        axs[i][j].invert_yaxis();
        



Fig.savefig(f'{figs}/Figures_raw/Subcluster_deconv_CXCL13_T_B_6samples_ScatterPlot.pdf', bbox_inches='tight')

In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(2,4, figsize=(18,8), dpi=300)
samples = ['LUAD_A', 'TD5']
celltype = ['CXCL13_CD4_Tex', 'CXCL13_CD8_Tex', 'Naive_B', 'Memory_B']
celltype_vmax ={'Naive_B':0.3, 'Memory_B':0.3, 'CXCL13_CD4_Tex':0.1, 'CXCL13_CD8_Tex':0.1}

for i in range(2):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == samples[i]].copy(),
                        color=celltype[j],
                        basis = 'spatial',
                        size=40, vmax=celltype_vmax[celltype[j]],
                        add_outline=True, frameon = True, show=False, ax=axs[i][j]);
        
        axs[i][j].set_xticks([]);
        axs[i][j].set_yticks([]);
        axs[i][j].set_xlabel('')
        axs[i][j].set_ylabel('')
        axs[i][j].set_title('')
        axs[i][j].invert_yaxis();

Fig.savefig(f'{figs}/Figures_raw/Subcluster_deconv_CXCL13_T_B_ScatterPlot.pdf', bbox_inches='tight')


### CXCL13_T_Tex Spots Visulization

In [ ]:
# CXCL13_CD8_Tex positive spots
all_8samples_st_ssGSEA.obs['CXCL13_CD8_Tex_positive'] = ['Positive' if i > 0.03 else 'False' for i in all_8samples_st_ssGSEA.obs['CXCL13_CD8_Tex']]

# CXCL13_CD4_Tex positive spots
all_8samples_st_ssGSEA.obs['CXCL13_CD4_Tex_positive'] = ['Positive' if i > 0.03 else 'False' for i in all_8samples_st_ssGSEA.obs['CXCL13_CD4_Tex']]

In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(2,4, figsize=(18,8), dpi=150)
patients = [['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3'],
            ['TD1', 'TD5', 'TD6', 'TD8']]

for i in range(2):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patients[i][j]].copy(),
                        color=['CXCL13_CD8_Tex_positive'],
                        basis = 'spatial',
                        title=f'{patients[i][j]}; CXCL13_CD8_Tex',
                        size=40, 
                        add_outline=True, frameon = True,
                        show=False, ax=axs[i][j]);
        # 移除图例
        axs[i][j].legend([], frameon=False)  # 确保图例被清空        
        axs[i][j].set_title('')
        axs[i][j].set_xticks([])
        axs[i][j].set_yticks([])
        axs[i][j].set_xlabel('')
        axs[i][j].set_ylabel('')
        axs[i][j].invert_yaxis();
        axs[i][j].set_facecolor('none');
        
Fig.patch.set_facecolor('none')
Fig.savefig(f'{figs}/Figures_raw/ST_CXCL13_CD8_Tex_positive_Region_Spatial_ScatterPlot.png', bbox_inches='tight', transparent = True)

In [ ]:
#Fig, axs = plt.Figure(figsize=(9,3), dpi=300)
Fig, axs = plt.subplots(2,4, figsize=(18,8), dpi=150)
patients = [['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3'],
            ['TD1', 'TD5', 'TD6', 'TD8']]

for i in range(2):
    for j in range(4):
        sc.pl.embedding(all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patients[i][j]],
                        color=['CXCL13_CD4_Tex_positive'],
                        basis = 'spatial',
                        title=f'{patients[i][j]}; CXCL13_CD4_Tex',
                        size=40, 
                        add_outline=True, frameon = True,
                        show=False, ax=axs[i][j]);

        # 移除图例
        axs[i][j].legend([], frameon=False)  # 确保图例被清空
        axs[i][j].set_title('')
        axs[i][j].set_xticks([])
        axs[i][j].set_yticks([])
        axs[i][j].set_xlabel('')
        axs[i][j].set_ylabel('')
        axs[i][j].invert_yaxis();
        axs[i][j].set_facecolor('none');
        
Fig.patch.set_facecolor('none')
Fig.savefig(f'{figs}/Figures_raw/ST_CXCL13_CD4_Tex_positive_Region_Spatial_ScatterPlot.png', bbox_inches='tight', transparent = True)

## Neighbor analysis (RCTD)(Accepted)

### CXCL13 CD4 Based NeighborHood Analysis

In [ ]:
cxcl13_cd4_based_neighbor_df = pd.DataFrame()

#old
for patient in ['TD1', 'TD5',
                'TD6', 'TD8']:
    neighbor_df = neighbor_measurement(adata=all_8samples_st_ssGSEA,
                                       base_cell_type='CXCL13_CD4_Tex',
                                       query_cell_type=['Naive_B', 'Memory_B'],
                                       slices_column='orig.ident',
                                       slices=patient,
                                       base_cell_type_spots_fraction_cutoff=0.03)

    neighbor_df['Age_type'] = 'Old'
    cxcl13_cd4_based_neighbor_df = pd.concat([cxcl13_cd4_based_neighbor_df, neighbor_df]) 


#young
for patient in ['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3']:
    neighbor_df = neighbor_measurement(adata=all_8samples_st_ssGSEA,
                                       base_cell_type='CXCL13_CD4_Tex',
                                       query_cell_type=['Naive_B', 'Memory_B'],
                                       slices_column='orig.ident',
                                       slices=patient,
                                       base_cell_type_spots_fraction_cutoff=0.03)

    neighbor_df['Age_type'] = 'Young'
    cxcl13_cd4_based_neighbor_df = pd.concat([cxcl13_cd4_based_neighbor_df, neighbor_df]) 

cxcl13_cd4_based_neighbor_df = cxcl13_cd4_based_neighbor_df.melt(id_vars = ['Slice', 'Base_cell_type', 'Age_type'],
                                                                 value_vars = ['Naive_B', 'Memory_B'],
                                                                 var_name='Query_Cell_Type',
                                                                 value_name='Deconv_Fraction')


In [ ]:
cxcl13_cd4_based_neighbor_df

In [ ]:
figures, ax = plt.subplots(figsize=(10,8))

sns.boxplot(data=cxcl13_cd4_based_neighbor_df, x='Query_Cell_Type', y='Deconv_Fraction', ax=ax, hue='Age_type', gap=0.1)

plt.title('The CXCL13_CD4_Tex Based Neighborhood Analysis');

In [ ]:
scipy.stats.mannwhitneyu(x=cxcl13_cd4_based_neighbor_df.query("Age_type == 'Young' & Query_Cell_Type == 'Naive_B'")['Deconv_Fraction'],
                         y=cxcl13_cd4_based_neighbor_df.query("Age_type == 'Old' & Query_Cell_Type == 'Naive_B'")['Deconv_Fraction'])

In [ ]:
scipy.stats.mannwhitneyu(x=cxcl13_cd4_based_neighbor_df.query("Age_type == 'Young' & Query_Cell_Type == 'Memory_B'")['Deconv_Fraction'],
                         y=cxcl13_cd4_based_neighbor_df.query("Age_type == 'Old' & Query_Cell_Type == 'Memory_B'")['Deconv_Fraction'])

In [ ]:
#save
cxcl13_cd4_based_neighbor_df.to_csv(f'{obj}/Neighborhood_Analysis/cxcl13_cd4_based_neighbor_df.csv', index=False)

## CCC analysis (COMMOT)

In [ ]:
import commot as ct

In [ ]:
all_8samples_st_ssGSEA

In [ ]:
LR=np.array([['CXCL13', 'CXCR5', 'CXCL13_CXCR5_pathway']],dtype=str)
df_ligrec = pd.DataFrame(data=LR)

### CXCL13_CXCR5 interaction plot

In [ ]:
all_8samples_st_ssGSEA

In [ ]:
LR=np.array([['CXCL13', 'CXCR5', 'CXCL13_CXCR5_pathway']],dtype=str)
df_ligrec = pd.DataFrame(data=LR)

In [ ]:
Fig, axs = plt.subplots(8,2, figsize=(4.4,18))
patients = ['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3', 'TD1', 'TD5', 'TD6', 'TD8']

for i in range(8):
    adata = all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patients[i]].copy()
    ct.tl.spatial_communication(adata, database_name='user_database', df_ligrec=df_ligrec, dis_thr=500, heteromeric=True, pathway_sum=True)
    pts = adata.obsm['spatial']
    s = adata.obsm['commot-user_database-sum-sender']['s-CXCL13-CXCR5']
    r = adata.obsm['commot-user_database-sum-receiver']['r-CXCL13-CXCR5']

    # Sender
    axs[i][0].scatter(pts[:,0], pts[:,1], c=s, s=0.3, cmap='Reds')
    axs[i][0].set_title(F'Sender(CXCL13)')
    axs[i][0].set_xticks([])
    axs[i][0].set_yticks([])
    axs[i][0].set_xlabel('')
    axs[i][0].set_ylabel('')
    axs[i][0].invert_yaxis()

    # Receiver
    axs[i][1].scatter(pts[:,0], pts[:,1], c=r, s=0.3, cmap='Blues')
    axs[i][1].set_title(f'Receiver(CXCR5) {patients[i]}')
    axs[i][1].set_xticks([])
    axs[i][1].set_yticks([])
    axs[i][1].set_xlabel('')
    axs[i][1].set_ylabel('')
    axs[i][1].invert_yaxis();
        
        
    #axs[i][j].set_xticks([]);
    #axs[i][j].set_yticks([]);
    #axs[i][j].set_xlabel('')
    #axs[i][j].set_ylabel('')
    #axs[i][j].set_title('')
    #axs[i][j].invert_yaxis();

Fig.savefig(f'{figs}/Figures_raw/ST_CXCL13_CXCR5_Interaction_Plot_ScatterPlot.pdf', bbox_inches='tight')

### CCC_Score analysis

In [ ]:
ccc_score_df = pd.DataFrame()
patients = ['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3', 'TD1', 'TD5', 'TD6', 'TD8']
for patient in patients:
    adata = all_8samples_st_ssGSEA[all_8samples_st_ssGSEA.obs['orig.ident'] == patient].copy()
    ct.tl.spatial_communication(adata, database_name='user_database', df_ligrec=df_ligrec, dis_thr=500, heteromeric=True, pathway_sum=True)
    ccc_score = adata.obsp['commot-user_database-CXCL13-CXCR5'].toarray().flatten()
    #filter
    ccc_score = ccc_score[ccc_score > 0]

    #normalization
    ccc_score = ccc_score/adata.shape[0] * 1000
    ccc_score = pd.DataFrame(ccc_score, columns=['CXCL13_CXCR5_score'])
    ccc_score['Patient'] = patient
    if patient in ['LUAD_A', 'LUAD_B', 'LUAD_C', 'TD3']:
        ccc_score['Age_type'] = 'Young'
    else:
        ccc_score['Age_type'] = 'Old'
    ccc_score_df = pd.concat([ccc_score_df, ccc_score])

In [ ]:
ccc_score_df.shape

In [ ]:
ccc_score_df["log_CXCL13_CXCR5_score"] = np.log10(ccc_score_df["CXCL13_CXCR5_score"] + 1e-6)

In [ ]:
# 使用 MinMaxScaler 归一化
scaler = MinMaxScaler(feature_range=(0, 1))
ccc_score_df["normalized_log_CXCL13_CXCR5_score"] = scaler.fit_transform(ccc_score_df[["log_CXCL13_CXCR5_score"]])

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(x="Age_type", y="log_CXCL13_CXCR5_score", data=ccc_score_df, palette="viridis")

#plt.ylim(0, 0.05)  # 根据需要调整 y 轴范围
plt.title("CXCL13_CXCR5 Score by Age Type")
plt.xlabel("Age Type")
plt.ylabel("CXCL13_CXCR5 Score")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
sns.boxplot(x="Age_type", y="normalized_log_CXCL13_CXCR5_score", data=ccc_score_df, palette="viridis")

#plt.ylim(0, 0.05)  # 根据需要调整 y 轴范围
plt.title("CXCL13_CXCR5 Score by Age Type")
plt.xlabel("Age Type")
plt.ylabel("CXCL13-CXCR5 Interaction Score")
plt.xticks(rotation=45)
plt.tight_layout()

plt.savefig(f'{figs}/Figures_raw/ST_CXCL13_CXCR5_Interaction_Score_BoxPlot.pdf', bbox_inches='tight', transparent = True)

In [ ]:
scipy.stats.mannwhitneyu(x=ccc_score_df.query("Age_type == 'Young'")['normalized_log_CXCL13_CXCR5_score'],
                         y=ccc_score_df.query("Age_type == 'Old'")['normalized_log_CXCL13_CXCR5_score'])